# 07 反应优化入门：Buchwald-Hartwig 条件搜索

本节使用真实 Buchwald-Hartwig 数据，说明反应优化为什么是一个“有限预算下的序贯决策”问题。

应用场景：真实实验昂贵时，化学家希望在有限预算内尽快找到高产率条件。
本节数据来自 Ahneman、Estrada、Lin、Dreher 和 Doyle 关于 C-N cross-coupling 反应性能预测的 Science 2018 工作。

Reference:

- Ahneman, D. T.; Estrada, J. G.; Lin, S.; Dreher, S. D.; Doyle, A. G. Predicting reaction performance in C-N cross-coupling using machine learning. Science, 2018, 360, 186-190.

## 直觉解释

反应产率受底物、ligand、base、additive、aryl halide 等条件影响。
真实实验预算有限，因此我们希望用少量已观察实验训练 surrogate model，建议下一组更可能高产率的条件。

化学解释上，ligand 影响金属中心电子/空间环境，base 和 additive 影响反应路径和副反应，
aryl halide 改变底物反应性。模型只能从已有表格中学习这些关联，不能自动保证建议条件有机制合理性。

## 数学/化学定义

简单优化循环：

```text
1. 先观察少量实验 D = {(condition_i, yield_i)}
2. 训练 surrogate: condition -> yield
3. acquisition 选择下一组未做条件
4. 读取或执行该实验，更新 D
```

本节比较三类策略：

1. random search: 不学习，只随机抽条件。
2. Random Forest greedy: 用 one-hot 条件训练 RF surrogate，只选预测均值最高的条件。
3. Gaussian Process UCB: 用 GP surrogate 同时利用预测均值和不确定性，选择 `mean + beta * std` 最大的条件。

为了更接近实际高通量实验，本节使用 batch optimization：每轮一次选择 5 个候选条件，等这一批结果都观察到之后再训练下一轮模型。

包括三种初始设计：

- random initial design: 第一批随机选 5 个条件。
- Latin-like level coverage: 在离散条件空间中尽量覆盖每个因子的不同水平，类似 LHC 想覆盖各维度的思想。
- CVT-like/maximin: 在 one-hot 空间中贪心选择彼此尽量远的点，类似 CVT/space-filling design 的覆盖思想。

注意：严格的 LHC 和 CVT 主要用于连续变量空间。这里的反应条件是离散类别变量，所以使用的是面向类别空间的近似版本。

因此图中会有 7 条主要曲线：1 条 random baseline，加上 `3 initial designs × 2 surrogate models`。
每条曲线都跑 10 个 repeat，用均值和 10-90% 区间显示，避免把单次随机种子的偶然结果当成结论。

## 读取 Buchwald-Hartwig 数据

这一格读取反应条件表并把 yield 转成数值。每一行是一个已测过的条件组合；
模拟会从固定数据表中“读取结果”，不是真正执行新的湿实验。

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, DotProduct, WhiteKernel
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style="whitegrid", context="notebook")

# 原始表中每行是一组 reaction + ligand/base/additive/aryl halide 条件和对应 yield。
bh = pd.read_csv(RAW / "buchwald_hartwig.csv")
bh["yield"] = pd.to_numeric(bh["yield"], errors="coerce")
bh = bh.dropna(subset=["yield"]).reset_index(drop=True)
print(bh.shape)
bh.head()

## 定义完整候选空间

`reaction` 表示底物/反应身份，ligand、additive、base、aryl halide 表示可选反应条件。
如果同一个条件组合有重复记录，这里取平均 yield，避免同一实验重复影响优化轨迹。

In [ ]:
condition_columns = ["reaction", "ligand", "additive", "base", "aryl halide"]

bh_conditions = bh[condition_columns + ["yield"]].dropna().copy()
for col in condition_columns:
    bh_conditions[col] = bh_conditions[col].astype(str)

# 一个候选点就是 reaction + 条件组合；重复测量时用平均产率表示该候选点。
candidate_df = (
    bh_conditions
    .groupby(condition_columns, as_index=False, sort=False)["yield"]
    .mean()
    .reset_index(drop=True)
)

# 统计每类条件有多少种选择，用于量化组合空间为什么很快变大。
space_summary = pd.DataFrame(
    {
        "condition": condition_columns,
        "unique_choices": [candidate_df[col].nunique() for col in condition_columns],
    }
)
estimated_factorial = int(np.prod(space_summary["unique_choices"]))
print("realized candidate rows:", len(candidate_df))
print("full factorial combinations if fully crossed:", f"{estimated_factorial:,}")
space_summary

## 查看产率分布

如果高产率条件很少，随机搜索需要较大预算才有机会撞到好条件；
如果高产率条件很多，随机搜索也可能表现不错。

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(candidate_df["yield"], bins=30)
plt.xlabel("yield")
plt.title("Buchwald-Hartwig yield distribution across realized candidates")
plt.tight_layout()

## 高产率条件到底稀不稀有？

如果高产率条件并不稀有，random search 是很强的 baseline；这也是很多优化论文必须认真报告随机基线的原因。
下面先看不同产率阈值以上的候选比例。

In [ ]:
y_all = candidate_df["yield"].to_numpy(dtype=float)
thresholds = [40, 50, 60, 70, 80]

pd.DataFrame(
    {
        "yield_threshold": thresholds,
        "fraction_of_candidates_at_or_above": [(y_all >= t).mean() for t in thresholds],
        "count": [(y_all >= t).sum() for t in thresholds],
    }
)

## 编码反应条件

scikit-learn 模型不能直接处理字符串类别。one-hot encoding 会把每个类别水平变成一个 0/1 特征：
例如某一行是否使用了某个 ligand、某个 base、某个 reaction id。

这是一种很朴素的表示。它能表达“某个条件是否出现”，但不会显式表达配位化学、电子效应或底物结构相似性。

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_all = encoder.fit_transform(candidate_df[condition_columns].astype(str))
y_all = candidate_df["yield"].to_numpy(dtype=float)

print("X shape:", X_all.shape)
print("yield range:", f"{y_all.min():.1f} to {y_all.max():.1f}")
candidate_df.nlargest(5, "yield")[condition_columns + ["yield"]]

## 定义随机搜索和初始设计

反应优化中，前几组实验通常不是随便选的：需要尽量覆盖不同 ligand、base、additive 或底物类型。
下面几个函数分别给出 batch random baseline、random/Latin-like/CVT-like 初始化。

In [ ]:
def batch_best_trace(yields, observed_order, batch_size=5):
    # 在每个 batch 完成后记录“目前见过的最好产率”。
    observed_order = list(map(int, observed_order))
    counts = []
    best_values = []
    for end in range(batch_size, len(observed_order) + 1, batch_size):
        counts.append(end)
        best_values.append(float(np.max(yields[observed_order[:end]])))

    # 如果最后一个 batch 不满 batch_size，也记录最终结果。
    if counts and counts[-1] != len(observed_order):
        counts.append(len(observed_order))
        best_values.append(float(np.max(yields[observed_order])))
    elif not counts and observed_order:
        counts.append(len(observed_order))
        best_values.append(float(np.max(yields[observed_order])))

    return np.asarray(counts), np.asarray(best_values)


def simulate_random_batches(yields, budget=40, batch_size=5, repeats=200, seed=RANDOM_STATE):
    # Baseline: 每一轮随机选 batch_size 个未观察候选；下一轮仍然随机，不训练模型。
    # 这个过程只利用 observed set 来避免重复选择，不利用 observed yield 来指导下一轮。
    rng = np.random.default_rng(seed)
    traces = []
    n = len(yields)
    budget = min(budget, n)
    for _ in range(repeats):
        order = rng.choice(n, size=budget, replace=False)
        _, trace = batch_best_trace(yields, order, batch_size=batch_size)
        traces.append(trace)
    return np.asarray(traces)


def random_initial_indices(total_size, size=5, seed=RANDOM_STATE):
    # 初始 batch 的随机设计。
    rng = np.random.default_rng(seed)
    return list(map(int, rng.choice(total_size, size=min(size, total_size), replace=False)))


def latin_like_initial_indices(df, columns, size=8, seed=RANDOM_STATE):
    # 离散版“Latin-like”初始化：每一步优先选择能覆盖更多未见类别水平的候选。
    rng = np.random.default_rng(seed)
    selected = []
    used_values = {col: set() for col in columns}
    pool = list(rng.permutation(len(df)))

    while len(selected) < min(size, len(df)) and pool:
        gains = []
        for idx in pool:
            row = df.iloc[idx]
            gains.append(sum(row[col] not in used_values[col] for col in columns))

        # gain 相同的时候随机打破平局，避免总是受原始表顺序影响。
        best_gain = max(gains)
        tied_positions = [pos for pos, gain in enumerate(gains) if gain == best_gain]
        chosen_position = int(rng.choice(tied_positions))
        idx = int(pool.pop(chosen_position))
        selected.append(idx)
        for col in columns:
            used_values[col].add(df.iloc[idx][col])

    return selected


def maximin_initial_indices(X, size=8, seed=RANDOM_STATE):
    # CVT-like / space-filling 思路：在 one-hot 空间中贪心选择离已有点最远的候选。
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    size = min(size, n)
    first = int(rng.integers(n))
    selected = [first]
    min_dist = np.full(n, np.inf)

    while len(selected) < size:
        last = selected[-1]
        diff = X - X[last]
        dist = np.sqrt(np.einsum("ij,ij->i", diff, diff))
        min_dist = np.minimum(min_dist, dist)
        min_dist[selected] = -np.inf
        selected.append(int(np.argmax(min_dist)))

    return selected

## 定义 surrogate 优化循环

RF greedy 只看预测均值，容易过早 exploitation。
GP-UCB 使用预测均值和不确定性，形式是：

```text
acquisition(x) = mean(x) + beta * std(x)
```

`beta` 越大，越鼓励探索模型不确定的区域。

这里每一轮选择一个 batch 的候选点。注意：同一个 batch 中的 5 个点是同时选择的，
不会因为第 1 个点结果很好就立刻改变第 2-5 个点。

In [ ]:
def _prepare_observed(init_indices, total_size, limit):
    # 去重并保证初始化数量不超过 batch size、预算和候选数量。
    observed = []
    for idx in init_indices:
        idx = int(idx)
        if idx not in observed:
            observed.append(idx)
        if len(observed) >= min(total_size, limit):
            break
    return observed


def _take_top_k(remaining, scores, k):
    # 从 remaining 中取 acquisition score 最高的 k 个候选，作为下一批实验。
    order = np.argsort(scores)[::-1][:k]
    return [remaining[int(pos)] for pos in order]


def run_rf_batch_greedy(X, y, init_indices, budget=40, batch_size=5, seed=RANDOM_STATE):
    observed = _prepare_observed(init_indices, len(y), min(batch_size, budget))
    remaining = [idx for idx in range(len(y)) if idx not in observed]

    while len(observed) < min(budget, len(y)) and remaining:
        model = RandomForestRegressor(
            n_estimators=60,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(X[observed], y[observed])
        predicted_mean = model.predict(X[remaining])

        # Greedy acquisition: 每轮一次性选择预测均值最高的一批候选。
        k = min(batch_size, budget - len(observed), len(remaining))
        next_batch = _take_top_k(remaining, predicted_mean, k)
        observed.extend(next_batch)
        remaining = [idx for idx in remaining if idx not in set(next_batch)]
    counts, trace = batch_best_trace(y, observed, batch_size=batch_size)
    return counts, trace, observed


def run_gp_batch_ucb(X, y, init_indices, budget=40, batch_size=5, beta=1.5, seed=RANDOM_STATE):
    observed = _prepare_observed(init_indices, len(y), min(batch_size, budget))
    remaining = [idx for idx in range(len(y)) if idx not in observed]

    while len(observed) < min(budget, len(y)) and remaining:
        # DotProduct kernel 在 one-hot 类别特征上相当于学习线性相似性；
        # WhiteKernel 表示观测噪声。这里关闭超参数优化，使运行更快、更稳定。
        kernel = ConstantKernel(1.0) * DotProduct(sigma_0=1.0) + WhiteKernel(noise_level=1.0)
        model = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            alpha=1e-6,
            optimizer=None,
            random_state=seed,
        )
        model.fit(X[observed], y[observed])
        predicted_mean, predicted_std = model.predict(X[remaining], return_std=True)

        # UCB acquisition: exploitation + exploration。
        acquisition = predicted_mean + beta * predicted_std
        k = min(batch_size, budget - len(observed), len(remaining))
        next_batch = _take_top_k(remaining, acquisition, k)
        observed.extend(next_batch)
        remaining = [idx for idx in remaining if idx not in set(next_batch)]
    counts, trace = batch_best_trace(y, observed, batch_size=batch_size)
    return counts, trace, observed

## 比较搜索策略

曲线表示每一轮 batch 完成后，当前已经观察到的最好产率。黑色虚线是这个固定数据表中的全局最好值；
真实实验中我们通常不知道这条线在哪里。

这里的 random baseline 是“每轮随机选 5 个未做条件，下一轮仍然随机”。它不训练模型，
但会避免重复选择同一个候选。surrogate 方法则是先用 random/Latin-like/CVT-like 选第一批 5 个点，
之后每轮用 RF 或 GP-UCB 选择下一批 5 个点。

为了避免偶然性，每个策略跑 10 个 repeat。横轴显示 round：round 1 是第一批 5 个实验完成后，
round 20 对应总共 100 个实验完成后。

如果 random search 和 surrogate 很接近，可能有几个原因：

1. 候选空间不大，预算占比不低。
2. 高产率候选在空间里并不稀有。
3. one-hot 类别特征很粗糙，RF/GP 不能真正理解反应机理。
4. 小数据早期 surrogate 很不稳定；没有足够观测时，模型建议未必优于随机。

In [ ]:
batch_size = 5
n_rounds = 20
n_repeats = 10
budget = min(batch_size * n_rounds, len(candidate_df))
actual_rounds = int(np.ceil(budget / batch_size))
round_steps = np.arange(1, actual_rounds + 1)

def make_initial_design(design_name, seed):
    if design_name == "random init":
        return random_initial_indices(len(candidate_df), size=batch_size, seed=seed)
    if design_name == "Latin-like init":
        return latin_like_initial_indices(candidate_df, condition_columns, size=batch_size, seed=seed)
    if design_name == "CVT-like init":
        return maximin_initial_indices(X_all, size=batch_size, seed=seed)
    raise ValueError(f"unknown initial design: {design_name}")

initial_design_names = ["random init", "Latin-like init", "CVT-like init"]
random_traces = simulate_random_batches(
    y_all,
    budget=budget,
    batch_size=batch_size,
    repeats=n_repeats,
    seed=RANDOM_STATE,
)

method_results = {}
for design_name in initial_design_names:
    rf_traces = []
    rf_observed_runs = []
    gp_traces = []
    gp_observed_runs = []

    for repeat in range(n_repeats):
        seed = RANDOM_STATE + repeat
        init_indices = make_initial_design(design_name, seed)

        _, trace, observed = run_rf_batch_greedy(
            X_all, y_all, init_indices, budget=budget, batch_size=batch_size, seed=seed
        )
        rf_traces.append(trace)
        rf_observed_runs.append(observed)

        _, trace, observed = run_gp_batch_ucb(
            X_all, y_all, init_indices, budget=budget, batch_size=batch_size, beta=1.5, seed=seed
        )
        gp_traces.append(trace)
        gp_observed_runs.append(observed)

    rf_traces = np.asarray(rf_traces)
    gp_traces = np.asarray(gp_traces)
    rf_best_run = int(np.argmax(rf_traces[:, -1]))
    gp_best_run = int(np.argmax(gp_traces[:, -1]))

    method_results[f"RF + {design_name}"] = {
        "model": "RF greedy",
        "initial_design": design_name,
        "traces": rf_traces,
        "best_observed": rf_observed_runs[rf_best_run],
    }

    method_results[f"GP-UCB + {design_name}"] = {
        "model": "GP-UCB",
        "initial_design": design_name,
        "traces": gp_traces,
        "best_observed": gp_observed_runs[gp_best_run],
    }

plt.figure(figsize=(9, 5))
plt.plot(round_steps, random_traces.mean(axis=0), label="random baseline", linewidth=2.5, color="black")
plt.fill_between(
    round_steps,
    np.percentile(random_traces, 10, axis=0),
    np.percentile(random_traces, 90, axis=0),
    color="black",
    alpha=0.12,
    label="random 10-90%",
)

for label, result in method_results.items():
    linestyle = "--" if result["model"] == "RF greedy" else "-"
    traces = result["traces"]
    mean_trace = traces.mean(axis=0)
    p10 = np.percentile(traces, 10, axis=0)
    p90 = np.percentile(traces, 90, axis=0)
    line, = plt.plot(round_steps, mean_trace, label=label, linewidth=2, linestyle=linestyle)
    plt.fill_between(round_steps, p10, p90, color=line.get_color(), alpha=0.08)

plt.axhline(y_all.max(), linestyle="--", color="black", linewidth=1, label="best in dataset")
plt.xlabel("optimization round")
plt.ylabel("best yield observed so far")
plt.title(f"Batch reaction optimization: {batch_size} experiments/round, {n_repeats} repeats")
plt.xticks(round_steps)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## 数值汇总

把随机搜索和 6 个 surrogate 组合在 10 个 repeat 中的最终最好值放在一起。
这里的重点不是证明某个方法永远更好，而是学会用 mean 和 uncertainty band 判断 surrogate loop 到底有没有带来增益。

如果单次运行中 `RF + random init` 最好，不一定说明它稳定优于 GP 或 space-filling init。
RF 对 one-hot 离散类别表格通常很强；GP-UCB 的不确定性在高维稀疏 one-hot 空间中也可能校准不好。
因此需要看 10 次 repeat 的平均表现和 10-90% 区间。

In [ ]:
random_final = random_traces[:, -1]

summary_rows = [
    {
        "method": "random baseline",
        "initial_design": "random every batch",
        "mean_best_yield_after_20_rounds": random_final.mean(),
        "p10": np.percentile(random_final, 10),
        "p90": np.percentile(random_final, 90),
        "best_repeat": random_final.max(),
    }
]

for label, result in method_results.items():
    final_values = result["traces"][:, -1]
    summary_rows.append(
        {
            "method": result["model"],
            "initial_design": result["initial_design"],
            "mean_best_yield_after_20_rounds": final_values.mean(),
            "p10": np.percentile(final_values, 10),
            "p90": np.percentile(final_values, 90),
            "best_repeat": final_values.max(),
        }
    )

method_summary = pd.DataFrame(summary_rows).sort_values("mean_best_yield_after_20_rounds", ascending=False)
method_summary

## 查看 surrogate 实际观察到的好条件

最后列出 6 个 surrogate 组合各自 best repeat 中观察过的最高产率条件。分析重点不是“模型赢了没有”，
而是这些条件是否有化学合理性，以及如果真实实验昂贵，下一步是否还需要探索多样性。

In [ ]:
def best_observed_conditions(method_label, observed, top=3):
    out = candidate_df.iloc[observed].copy()
    out["order"] = np.arange(1, len(out) + 1)
    out["method"] = method_label
    return out.sort_values("yield", ascending=False).head(top)[["method", "order", *condition_columns, "yield"]]


pd.concat(
    [best_observed_conditions(label, result["best_observed"]) for label, result in method_results.items()],
    ignore_index=True,
)

## 观察问题

1. 为什么只固定一个 reaction id 时，surrogate loop 可能和 random search 差不多？
2. batch size = 5 时，为什么同一个 batch 内不能用第一个点的结果更新后四个点？
3. RF greedy 和 GP-UCB 的 acquisition 有什么本质差别？
4. LHC/CVT 这类 space-filling design 为什么在连续变量中更自然？离散反应条件中应该怎么解释？
5. 如果单次运行中 `RF + random init` 最好，应该怎样判断这是偶然还是稳定优势？
6. 如果模型建议的条件不符合化学常识，应该相信模型还是重新检查数据和假设？

### Hints

1. 固定 reaction id 后空间很小；如果高产率条件比例不低，random 很容易撞到好条件。RF 早期数据少、one-hot 表示粗糙，也不一定能学出可靠规律。
2. batch 实验是并行提交的一组实验；只有整批完成后才知道产率，所以同一批次中不能序贯更新模型。
3. RF greedy 只利用预测均值，倾向 exploitation；GP-UCB 加入预测标准差，能把不确定区域也纳入考虑。
4. 连续空间可以均匀填充几何区域；离散类别空间没有天然距离，所以这里只能用“覆盖水平”和“one-hot 距离”作为近似。
5. 看多个 repeat 的平均最终结果、分位区间和曲线形状；如果区间大量重叠，就不应该过度解释单次排名。
6. 先检查数据、编码、候选条件是否合理，再把建议当作可验证假设。模型不能替代化学约束和实验安全判断。

## 小结

Buchwald-Hartwig 数据让我们看到 AI4Chem 的另一个典型问题：不是预测单个分子性质，而是在有限实验预算下选择下一步实验。
random search 是必须认真比较的强 baseline；GP、RF、LHC/CVT-like 初始化都只是帮助我们更系统地提出实验假设。
surrogate model 是决策辅助工具，不是自动替代化学判断。